In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import jax
import numpy as np
#from scipy.interpolate import interp1d
from interpax import Interpolator1D

# TODO: this setup should be done taking into account the device description as a submodule

# Get sensor positions
with open("/home/zkeith/proj/ONW/ryan-onsim/Device-description/device_description/SPARC/240510/device.json") as f:
    device_description = json.load(f)

BpLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BP-LOMN')[0]
BnLowmnInd = np.where(np.array([x['name'][0:7] for x in device_description['magnetics']['b_field_pol_probe']]) == 'BN-LOMN')[0]

phi_probes = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BpLowmnInd]
phi_sens = [device_description['magnetics']['b_field_pol_probe'][i]['position']['phi'] for i in BnLowmnInd]

# Using 12 probe design from 2 years ago
# 8, 211, 91, 271, 288, 171, 228, 351, 131, 48, 328, 311
probe_order_indices = [0, 9, 3, 11, 12, 7, 10, 15, 5, 2, 14, 13]
probe_connection_indices = []
for i in range(len(probe_order_indices)):
    if i != len(probe_order_indices) - 1:
        probe_connection_indices.append([probe_order_indices[i], probe_order_indices[i+1]])
    else:
        probe_connection_indices.append([probe_order_indices[i], probe_order_indices[0]])

probe_connections = []
for connection in probe_connection_indices:
    probe_connections.append([phi_probes[connection[0]], phi_probes[connection[1]]])

# Get frequency responses
filepath = "../../modules/" 
fname = '21_mode_resp_data.txt'
out = np.loadtxt(filepath + fname, skiprows=1)

freq = out[:,0]
Bp_per_A = out[:,1] # Bp (poloidal field) per Amp of tearing mode current
Br_per_A = out[:,2] # Br (radial field) per Amp of tearing mode current

# we also want to build interpolated functions that can be called for 
# any frequency in order to get the br and bp
jax.config.update("jax_platforms", "cpu")
func_Bp_per_A = Interpolator1D(freq, Bp_per_A) # name stands for "function, bp per A"
func_Br_per_A = Interpolator1D(freq, Br_per_A) # name stands for "function, br per A"

In [33]:
import popsim.param_utils as param_utils
from popsim.modules.tearing import DisruptionPhase, IslandRotationPhase, Island, Tearing

# Set up tearing module
dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
tearing_config = Tearing.Config(
    magx_time=time_base
)

def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def generate_disruption_phase_trajectory(trigger_time: float, tq_to_cq_dur: float):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    disrupt_phase_dict = {
        0.0: DisruptionPhase.NONE,
        trigger_time - dt: DisruptionPhase.NONE,
        trigger_time: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur - dt: DisruptionPhase.TQ,
        trigger_time + tq_to_cq_dur: DisruptionPhase.CQ,
    }

    # Round the times to the nearest time step in the time base.
    disrupt_phase_dict = {
        find_nearest(time_base, time): phase
        for time, phase in disrupt_phase_dict.items()
    }
    return disrupt_phase_dict

def generate_island_rotation_phase_trajectory(
    trigger_time: float, rot_dur: float, locking_dur: float
):
    # TODO(allenw): we want a rectilinear interpolation scheme.
    rot_phase_dict = {
        0.0: IslandRotationPhase.NONE,
        trigger_time - dt: IslandRotationPhase.NONE,
        trigger_time: IslandRotationPhase.SPAWN,
        trigger_time + dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur - dt: IslandRotationPhase.ROTATING,
        trigger_time + rot_dur: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur - dt: IslandRotationPhase.DECELERATING,
        trigger_time + rot_dur + locking_dur: IslandRotationPhase.LOCKED,
    }

    # Round the times to the nearest time step in the time base.
    rot_phase_dict = {
        find_nearest(time_base, time): phase for time, phase in rot_phase_dict.items()
    }
    return rot_phase_dict

islands = [Island(3,2), Island(2,1)]

W = {island: 0.0 for island in islands}
F = {island: 0.0 for island in islands}
mode_phase={island: 0.0 for island in islands}

tearing_initial_state = Tearing.State(W=W, F=F, mode_phase=mode_phase)

rot_dur = 1.0
trigger_time = 0.0001
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

tearing_params = Tearing.Params(
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike
    ),
    island_rotation_phase=generate_island_rotation_phase_trajectory(
        trigger_time, rot_dur, locking_dur
    ),
)

tearing_module = Tearing(config=tearing_config, islands=islands)

In [34]:
import jax
from popsim.simulate import simulate
from popsim.modules.magnetic_measurements import LowNArray

lown_array_config = LowNArray.Config(
    probe_connections=probe_connections,
)

lown_array_initial_state = LowNArray.State(tearing_state=tearing_initial_state)

lown_array_params = LowNArray.Params(
    tearing_params=tearing_params,
)

lown_array_module = LowNArray(config=lown_array_config, tearing_module=tearing_module, func_Bp_per_A=func_Bp_per_A)

jax.config.update("jax_platforms", "cpu")
lown_xarray = simulate(lown_array_module, time_base, lown_array_initial_state, lown_array_params)

Unique mode numbers: [1 2]
Design Matrix: [[ 1.84165603e+00  6.70307976e-01  5.02314256e-01 -5.98634819e-01]
 [-8.26463216e-01 -1.52215589e+00  1.45262003e+00  9.43342480e-01]
 [-5.23538966e-02  1.99931465e+00  0.00000000e+00  1.94289029e-16]
 [-2.91127708e-01 -5.13336698e-02 -1.99994025e-01  5.49479067e-01]
 [ 1.30632052e+00 -1.09613307e+00 -1.75494027e+00 -3.09443318e-01]
 [-3.26395815e-01  8.96765132e-01  1.07817410e+00 -1.28491786e+00]
 [-1.65163591e+00 -6.01146310e-01 -1.07817410e+00  1.28491786e+00]
 [ 1.65163591e+00 -8.96765132e-01  1.07817410e+00  7.00174447e-01]
 [-1.32524010e+00  0.00000000e+00  1.94289029e-16 -1.98509230e+00]
 [-1.90020116e-01  1.27145429e+00 -5.75859843e-01  1.88355268e+00]
 [ 1.90020116e-01  2.26457156e-01  5.75859843e-01  1.01539627e-01]
 [-3.26395815e-01 -8.96765132e-01 -1.07817410e+00 -1.28491786e+00]]


In [35]:
from popsim.visualize import visualize_time_series

lown_xarray.to_netcdf("lown_simulation.nc")
#visualize_time_series(lown_xarray, max_cols=2)